In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '隐藏代码 [Hide]' : '显示代码 [Show]';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">显示代码 [Show]</button>
</div>
'''))


# 第4章 处理器基础 - 取指-执行周期 (Fetch-Execute Cycle)

## Cambridge International AS & A Level Computer Science (9618)

---

> **核心问题**：CPU每秒执行数十亿条指令，它到底是怎么做到的？每条指令经历了怎样的旅程？

取指-执行周期（Fetch-Execute Cycle，也叫 Fetch-Decode-Execute Cycle）是计算机运行的最基本机制。
**计算机从开机到关机，CPU一直在不断重复这个周期。**

## 1. 整体概览：三个阶段

每条指令都要经历三个阶段：

```
┌─────────┐    ┌──────────┐    ┌──────────┐
│  FETCH   │───→│  DECODE  │───→│ EXECUTE  │
│  取指    │    │  解码    │    │  执行    │
└─────────┘    └──────────┘    └──────────┘
     ↑                                │
     └────────────────────────────────┘
              不断循环
```

> **类比**：想象你是一个厨师在看食谱做菜：
> 1. **取指 (Fetch)**：眼睛看到食谱上的下一步指令
> 2. **解码 (Decode)**：大脑理解这个指令的含义
> 3. **执行 (Execute)**：双手完成这个操作
> 然后重复——看下一步、理解、执行...

## 2. 取指阶段 (Fetch Phase) — 详细步骤

这是最重要的阶段，需要你牢记每一步！

### 步骤1：PC中的地址复制到MAR
```
MAR ← [PC]
```
- PC（程序计数器）里存着下一条指令的地址
- 这个地址通过**地址总线**发送到内存
- MAR接收这个地址

### 步骤2：PC递增
```
PC ← [PC] + 1
```
- PC自动加1，指向下一条指令
- 这一步与内存访问**同时进行**，节省时间
- 为什么要这么早递增？因为等到执行阶段，CIR里可能有跳转指令会修改PC

### 步骤3：从内存读取指令到MDR
```
MDR ← [[MAR]]
```
- 内存在MAR指定的地址处找到指令
- 指令通过**数据总线**传输到MDR
- 双方括号 `[[MAR]]` 的含义：MAR中存的地址所指向的内存内容

### 步骤4：指令从MDR复制到CIR
```
CIR ← [MDR]
```
- 指令从MDR复制到CIR
- 现在CU可以开始解码这条指令了
- MDR被"释放"，可以在执行阶段用于数据传输

## 3. 解码阶段 (Decode Phase)

CU（控制单元）分析CIR中的指令：
- 识别**操作码 (Opcode)**：要执行什么操作？（加法？存储？跳转？）
- 识别**操作数 (Operand)**：操作的对象是什么？（哪个地址？哪个值？）

```
指令格式：[ Opcode | Operand ]
例如：      [ ADD   |  200   ]  → 将地址200的值加到ACC
```

## 4. 执行阶段 (Execute Phase)

根据解码的结果，CPU执行相应的操作：
- 如果是**算术/逻辑**指令 → ALU执行计算
- 如果是**数据传输**指令 → 从内存读取或写入数据
- 如果是**跳转**指令 → 修改PC的值
- 如果是**I/O**指令 → 与输入/输出设备交互

## 5. 寄存器传输记法 (Register Transfer Notation, RTN)

RTN是一种精确描述数据在寄存器之间移动的符号系统。**考试中经常考！**

### 符号说明

| 符号 | 含义 | 例子 |
|------|------|------|
| `←` | 赋值/传输 | `MAR ← [PC]` 意为"将PC的内容传输到MAR" |
| `[X]` | X中的内容 | `[PC]` 意为"PC里存储的值" |
| `[[X]]` | X中地址指向的内存内容 | `[[MAR]]` 意为"MAR里的地址所指向的内存位置中的内容" |

### 完整的取指周期RTN

```
MAR  ← [PC]        // 步骤1: 将PC的内容复制到MAR
PC   ← [PC] + 1    // 步骤2: PC递增
MDR  ← [[MAR]]     // 步骤3: 从MAR指定地址读取数据到MDR
CIR  ← [MDR]       // 步骤4: 将MDR的内容复制到CIR
```

### 执行阶段RTN示例

**LDD 200** (直接寻址加载):
```
MAR ← CIR(operand)  // 取出操作数部分（地址200）
MDR ← [[MAR]]       // 读取地址200的内容
ACC ← [MDR]          // 将数据放入累加器
```

**ADD 201** (加法):
```
MAR ← CIR(operand)  // 取出地址201
MDR ← [[MAR]]       // 读取地址201的内容
ACC ← [ACC] + [MDR]  // ACC = ACC + 数据
```

**STO 202** (存储):
```
MAR ← CIR(operand)  // 取出地址202
MDR ← [ACC]          // 将ACC的内容放入MDR
[[MAR]] ← [MDR]      // 将MDR的内容写入地址202
```

## 6. 取指-执行周期可视化

让我们用 matplotlib 创建一个分步骤的可视化：

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import matplotlib.patches as mpatches

steps = [
    ('Step 1: MAR \u2190 [PC]', 'PC\u4e2d\u7684\u5730\u5740\u590d\u5236\u5230MAR\n\u901a\u8fc7\u5730\u5740\u603b\u7ebf\u53d1\u9001', '#e94560'),
    ('Step 2: PC \u2190 [PC] + 1', 'PC\u81ea\u52a8\u52a01\n\u6307\u5411\u4e0b\u4e00\u6761\u6307\u4ee4', '#ffd700'),
    ('Step 3: MDR \u2190 [[MAR]]', '\u5185\u5b58\u4e2d\u7684\u6307\u4ee4\u901a\u8fc7\n\u6570\u636e\u603b\u7ebf\u4f20\u5230MDR', '#00d2ff'),
    ('Step 4: CIR \u2190 [MDR]', '\u6307\u4ee4\u4eceMDR\u590d\u5236\u5230CIR\nCU\u5f00\u59cb\u89e3\u7801', '#53d769'),
    ('Step 5: DECODE', 'CU\u89e3\u7801\u6307\u4ee4\n\u8bc6\u522b\u64cd\u4f5c\u7801\u548c\u64cd\u4f5c\u6570', '#ff9ff3'),
    ('Step 6: EXECUTE', 'ALU/CU\u6267\u884c\u6307\u4ee4\n\u7ed3\u679c\u5b58\u5165\u76f8\u5e94\u5bc4\u5b58\u5668', '#f368e0'),
]

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#1a1a2e')
ax.axis('off')

for i, (title, desc, color) in enumerate(steps):
    x = (i % 3) * 4.2 + 0.5
    y = 4 - (i // 3) * 3.5
    box = mpatches.FancyBboxPatch((x, y), 3.5, 2.8, boxstyle='round,pad=0.2',
        facecolor='#16213e', edgecolor=color, linewidth=2.5)
    ax.add_patch(box)
    ax.text(x+1.75, y+1.4, title, fontsize=10, fontweight='bold',
        color=color, ha='center', va='center')
    ax.text(x+1.75, y+1.4, desc, fontsize=9, color='#dfe6e9',
        ha='center', va='center', linespacing=1.5)
    # Number circle
    circle = plt.Circle((x+0.3, y+2.5), 0.25, color=color, alpha=0.8)
    ax.add_patch(circle)
    ax.text(x+0.3, y+1.4, str(i+1), fontsize=11, fontweight='bold',
        color='white', ha='center', va='center')
    # Arrow to next
    if i < len(steps) - 1:
        if i % 3 < 2:
            ax.annotate('', xy=(x+3.7, y+1.4), xytext=(x+3.5, y+1.4),
                arrowprops=dict(arrowstyle='->', color='white', lw=1.5))

ax.set_xlim(-0.5, 14)
ax.set_ylim(-1, 8)
plt.title('Fetch-Decode-Execute Cycle Steps',
    fontsize=14, fontweight='bold', color='white', pad=15)
plt.tight_layout()
plt.show()


## 7. 交互式取指-执行周期模拟器

下面是一个完整的FE周期模拟器，演示CPU如何执行一段简单程序：

In [ ]:
class FetchExecuteSimulator:
    def __init__(self):
        self.PC = 0
        self.MAR = 0
        self.MDR = ''
        self.CIR = ''
        self.ACC = 0
        self.IX = 0
        self.memory = {}
        self.step_log = []
    
    def load_program(self, start_addr, instructions, data=None):
        for i, inst in enumerate(instructions):
            self.memory[start_addr + i] = inst
        if data:
            for addr, val in data.items():
                self.memory[addr] = val
        self.PC = start_addr
    
    def log(self, msg):
        state = f'  PC={self.PC} MAR={self.MAR} MDR={self.MDR} CIR={self.CIR} ACC={self.ACC}'
        self.step_log.append(f'{msg}\n{state}')
        print(f'  >> {msg}')
        print(f'     PC={self.PC} | MAR={self.MAR} | MDR={self.MDR} | CIR={self.CIR} | ACC={self.ACC}')
    
    def fetch(self):
        print(f'\n--- FETCH ---')
        self.MAR = self.PC
        self.log('MAR <- [PC]')
        self.PC = self.PC + 1
        self.log('PC <- [PC] + 1')
        self.MDR = self.memory.get(self.MAR, 'NOP')
        self.log('MDR <- [[MAR]]')
        self.CIR = self.MDR
        self.log('CIR <- [MDR]')
    
    def decode_execute(self):
        inst = str(self.CIR)
        if inst == 'END' or inst == 'NOP':
            print('--- DECODE: END ---')
            return False
        parts = inst.split()
        opcode = parts[0]
        operand = int(parts[1]) if len(parts) > 1 else 0
        print(f'--- DECODE: opcode={opcode}, operand={operand} ---')
        print(f'--- EXECUTE ---')
        
        if opcode == 'LDD':
            self.MAR = operand
            self.MDR = self.memory.get(operand, 0)
            self.ACC = self.MDR
            self.log(f'LDD: ACC <- memory[{operand}] = {self.ACC}')
        elif opcode == 'LDM':
            self.ACC = operand
            self.log(f'LDM: ACC <- {operand}')
        elif opcode == 'ADD':
            self.MAR = operand
            self.MDR = self.memory.get(operand, 0)
            self.ACC = self.ACC + self.MDR
            self.log(f'ADD: ACC <- ACC + memory[{operand}] = {self.ACC}')
        elif opcode == 'SUB':
            self.MAR = operand
            self.MDR = self.memory.get(operand, 0)
            self.ACC = self.ACC - self.MDR
            self.log(f'SUB: ACC <- ACC - memory[{operand}] = {self.ACC}')
        elif opcode == 'STO':
            self.MAR = operand
            self.MDR = self.ACC
            self.memory[operand] = self.MDR
            self.log(f'STO: memory[{operand}] <- ACC = {self.ACC}')
        elif opcode == 'OUT':
            print(f'  ** OUTPUT: {self.ACC} **')
            self.log(f'OUT: display ACC = {self.ACC}')
        return True
    
    def run(self):
        print('=' * 60)
        print('Fetch-Execute Cycle Simulator')
        print('=' * 60)
        cycle = 1
        while True:
            print(f'\n{"=" * 40}')
            print(f'CYCLE {cycle}')
            print(f'{"=" * 40}')
            self.fetch()
            if not self.decode_execute():
                print('\n*** Program ended ***')
                break
            cycle += 1
            if cycle > 20:
                print('\n*** Safety limit reached ***')
                break

# Demo program: calculate (10 + 25) * 2 using shifts would be better
# but let's keep it simple: calculate 10 + 25 and output
sim = FetchExecuteSimulator()
sim.load_program(
    start_addr=100,
    instructions=['LDD 200', 'ADD 201', 'STO 202', 'OUT 0', 'END'],
    data={200: 10, 201: 25, 202: 0}
)
sim.run()
print(f'\n\u6700\u7ec8\u5185\u5b58[202] = {sim.memory[202]}')

## 8. 中断 (Interrupts)

### 什么是中断？

中断是一种机制，允许外部事件"打断"CPU正常的取指-执行周期。

> **类比**：你正在专心写作业（正常执行），突然电话响了（中断信号）。
> 你会：
> 1. 记住你做到了作业的哪一步（保存状态）
> 2. 去接电话（处理中断）
> 3. 打完电话后，回来继续做作业（恢复状态）

### 为什么需要中断？

如果没有中断机制，CPU必须不断地"轮询" (polling) 每个设备：
- "键盘，你有输入吗？" → 没有
- "鼠标，你移动了吗？" → 没有
- "硬盘，数据好了吗？" → 没有
- 这样CPU大量时间浪费在检查上！

有了中断：设备需要CPU注意时才发信号，CPU可以专心执行程序。

### 中断处理流程

```
正常执行 → 检查中断 → 发现中断 → 保存当前状态
    ↑                              ↓
    ←── 恢复状态 ←── 处理中断 ←──┘
```

详细步骤：
1. 完成当前指令的执行
2. 将所有寄存器的值（特别是PC）压入 **栈 (Stack)**
3. 将 **中断服务程序 (ISR, Interrupt Service Routine)** 的地址加载到PC
4. 执行ISR
5. ISR完成后，从栈中恢复所有寄存器的值
6. 继续执行被中断的程序

## 9. 中断的类型

### 硬件中断 (Hardware Interrupts)
- 由外部设备触发
- 例如：键盘按键、鼠标点击、网络数据到达、硬盘读取完成

### 软件中断 (Software Interrupts)
- 由程序触发
- 例如：系统调用 (System Call)、除以零错误、内存访问违规

### 定时器中断 (Timer Interrupts)
- 由系统时钟定期触发
- 用途：多任务处理（操作系统定期切换运行中的程序）

### 中断优先级 (Interrupt Priority)

不是所有中断都同等重要！

| 优先级 | 类型 | 例子 |
|--------|------|------|
| 最高 | 硬件故障 | 电源故障、内存错误 |
| 高 | I/O操作 | 磁盘读写完成 |
| 中 | 定时器 | 任务调度 |
| 低 | 软件请求 | 程序的系统调用 |

> **关键概念**：高优先级中断可以打断低优先级中断的处理！
> 就像急诊室：心脏骤停的病人优先于骨折的病人。

## 10. 中断处理可视化

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#1a1a2e')
ax.axis('off')

# Timeline
phases = [
    (0, 2.5, '\u6b63\u5e38\u6267\u884c\n\u7a0b\u5e8f A', '#53d769'),
    (2.5, 1, '\u4e2d\u65ad!\n\u4fdd\u5b58\u72b6\u6001', '#e94560'),
    (3.5, 3, '\u6267\u884c ISR\n(\u4e2d\u65ad\u670d\u52a1\u7a0b\u5e8f)', '#ffd700'),
    (6.5, 1, '\u6062\u590d\u72b6\u6001', '#00d2ff'),
    (7.5, 2.5, '\u7ee7\u7eed\u6267\u884c\n\u7a0b\u5e8f A', '#53d769'),
]

y_center = 2.5
for x, w, label, color in phases:
    box = mpatches.FancyBboxPatch((x+0.1, y_center-0.7), w-0.2, 1.4,
        boxstyle='round,pad=0.1', facecolor=color, alpha=0.3, edgecolor=color, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y_center, label, fontsize=10, fontweight='bold',
        color=color, ha='center', va='center')

# Arrow for timeline
ax.annotate('', xy=(10.5, 0.5), xytext=(0, 0.5),
    arrowprops=dict(arrowstyle='->', color='white', lw=2))
ax.text(5, 0.2, 'Time \u2192', fontsize=11, color='white', ha='center', va='center')

# Interrupt signal
ax.annotate('\u4e2d\u65ad\u4fe1\u53f7!', xy=(2.5, y_center+0.7), xytext=(2.5, y_center+2),
    fontsize=11, color='#e94560', fontweight='bold', ha='center',
    arrowprops=dict(arrowstyle='->', color='#e94560', lw=2))

# Stack illustration
stack_x = 11
ax.text(stack_x+0.8, 4.5, 'Stack (\u6808)', fontsize=10, fontweight='bold',
    color='#ffeaa7', ha='center')
stack_items = ['PC = 103', 'ACC = 42', 'CIR = ADD 201', 'MAR = 201']
for i, item in enumerate(stack_items):
    box = mpatches.FancyBboxPatch((stack_x, 3.8-i*0.7), 1.6, 0.5,
        boxstyle='round,pad=0.05', facecolor='#2d3436', edgecolor='#ffeaa7', linewidth=1)
    ax.add_patch(box)
    ax.text(stack_x+0.8, 4.05-i*0.7, item, fontsize=7, color='#ffeaa7', ha='center', va='center')

ax.set_xlim(-0.5, 13)
ax.set_ylim(-0.5, 5.5)
plt.title('Interrupt Handling Timeline', fontsize=14, fontweight='bold', color='white', pad=10)
plt.tight_layout()
plt.show()

## 11. 带中断检查的完整取指-执行周期

```
┌──────────────────────────────────┐
│          开始                     │
└──────────────┬───────────────────┘
               ↓
┌──────────────────────────────────┐
│  FETCH: 取指                     │
│  MAR ← [PC]                     │
│  PC ← [PC] + 1                  │
│  MDR ← [[MAR]]                  │
│  CIR ← [MDR]                    │
└──────────────┬───────────────────┘
               ↓
┌──────────────────────────────────┐
│  DECODE: CU解码CIR中的指令       │
└──────────────┬───────────────────┘
               ↓
┌──────────────────────────────────┐
│  EXECUTE: 执行指令               │
└──────────────┬───────────────────┘
               ↓
┌──────────────────────────────────┐
│  检查是否有中断？                 │
│  ├─ 是 → 保存状态 → 执行ISR     │
│  │       → 恢复状态              │
│  └─ 否 → 回到FETCH              │
└──────────────────────────────────┘
```

## 12. 进阶模拟器：带中断处理的FE周期

In [ ]:
import random

class AdvancedFESimulator:
    def __init__(self):
        self.PC = 0
        self.MAR = 0
        self.MDR = ''
        self.CIR = ''
        self.ACC = 0
        self.memory = {}
        self.stack = []
        self.interrupt_pending = False
        self.isr_addr = 500
    
    def load_program(self, start, instructions, data=None):
        for i, inst in enumerate(instructions):
            self.memory[start + i] = inst
        if data:
            for addr, val in data.items():
                self.memory[addr] = val
        # Simple ISR
        self.memory[500] = 'LDM 999'
        self.memory[501] = 'OUT 0'
        self.memory[502] = 'END_ISR'
        self.PC = start
    
    def save_state(self):
        self.stack.append({'PC': self.PC, 'ACC': self.ACC, 'CIR': self.CIR, 'MAR': self.MAR})
        print(f'  [INTERRUPT] \u4fdd\u5b58\u72b6\u6001\u5230\u6808: PC={self.PC}, ACC={self.ACC}')
    
    def restore_state(self):
        if self.stack:
            state = self.stack.pop()
            self.PC = state['PC']
            self.ACC = state['ACC']
            print(f'  [INTERRUPT] \u4ece\u6808\u6062\u590d\u72b6\u6001: PC={self.PC}, ACC={self.ACC}')
    
    def execute_instruction(self, inst_str):
        if inst_str in ('END', 'NOP'):
            return 'END'
        if inst_str == 'END_ISR':
            return 'END_ISR'
        parts = str(inst_str).split()
        op = parts[0]
        val = int(parts[1]) if len(parts) > 1 else 0
        if op == 'LDD':
            self.ACC = self.memory.get(val, 0)
        elif op == 'LDM':
            self.ACC = val
        elif op == 'ADD':
            self.ACC += self.memory.get(val, 0)
        elif op == 'SUB':
            self.ACC -= self.memory.get(val, 0)
        elif op == 'STO':
            self.memory[val] = self.ACC
        elif op == 'OUT':
            print(f'  ** OUTPUT: {self.ACC} **')
        return 'OK'
    
    def run(self):
        print('=== Advanced FE Simulator with Interrupts ===')
        cycle = 0
        in_isr = False
        while cycle < 30:
            cycle += 1
            # Fetch
            self.MAR = self.PC
            self.PC += 1
            self.MDR = self.memory.get(self.MAR, 'END')
            self.CIR = self.MDR
            print(f'\nCycle {cycle}: Fetch addr={self.MAR} -> "{self.CIR}"')
            
            # Decode & Execute
            result = self.execute_instruction(str(self.CIR))
            if result == 'END':
                print('*** Program END ***')
                break
            if result == 'END_ISR':
                print('  [ISR \u5b8c\u6210]')
                self.restore_state()
                in_isr = False
                continue
            
            # Check for interrupt (simulate random interrupt)
            if not in_isr and cycle == 3:
                print(f'\n  !!! \u4e2d\u65ad\u53d1\u751f! (\u5728\u7b2c{cycle}\u4e2a\u5468\u671f) !!!')
                self.save_state()
                self.PC = self.isr_addr
                in_isr = True
                print(f'  PC \u8df3\u8f6c\u5230 ISR \u5730\u5740: {self.isr_addr}')
        print('\n=== Simulation Complete ===')

sim2 = AdvancedFESimulator()
sim2.load_program(
    start=100,
    instructions=['LDD 200', 'ADD 201', 'ADD 202', 'STO 203', 'OUT 0', 'END'],
    data={200: 10, 201: 20, 202: 30, 203: 0}
)
sim2.run()

## 13. RTN 练习工具

测试你对寄存器传输记法的理解：

In [ ]:
import random

rtn_questions = [
    ('MAR \u2190 [PC]', 'PC\u7684\u5185\u5bb9\u590d\u5236\u5230MAR', '\u53d6\u6307\u7b2c1\u6b65\uff1a\u544aMAR\u8981\u8bbf\u95ee\u54ea\u4e2a\u5730\u5740'),
    ('PC \u2190 [PC] + 1', 'PC\u7684\u503c\u52a01', '\u53d6\u6307\u7b2c2\u6b65\uff1aPC\u6307\u5411\u4e0b\u4e00\u6761\u6307\u4ee4'),
    ('MDR \u2190 [[MAR]]', 'MAR\u6307\u5411\u7684\u5185\u5b58\u5185\u5bb9\u590d\u5236\u5230MDR', '\u53d6\u6307\u7b2c3\u6b65\uff1a\u4ecemem\u8bfb\u6307\u4ee4'),
    ('CIR \u2190 [MDR]', 'MDR\u7684\u5185\u5bb9\u590d\u5236\u5230CIR', '\u53d6\u6307\u7b2c4\u6b65\uff1a\u6307\u4ee4\u51c6\u5907\u89e3\u7801'),
    ('ACC \u2190 [ACC] + [MDR]', 'ACC\u52a0\u4e0aMDR\u7684\u503c\uff0c\u7ed3\u679c\u5b58\u56deACC', '\u52a0\u6cd5\u6307\u4ee4\u7684\u6267\u884c\u9636\u6bb5'),
    ('[[MAR]] \u2190 [MDR]', 'MDR\u7684\u5185\u5bb9\u5199\u5165MAR\u6307\u5411\u7684\u5185\u5b58\u5730\u5740', '\u5b58\u50a8\u6307\u4ee4STO\u7684\u6267\u884c\u9636\u6bb5'),
]

print('=== RTN \u7ec3\u4e60 ===')
print()
random.shuffle(rtn_questions)
for rtn, meaning, context in rtn_questions:
    print(f'RTN: {rtn}')
    print(f'  \u542b\u4e49: {meaning}')
    print(f'  \u573a\u666f: {context}')
    print()

## 14. FE周期数据流向图

下面的图展示了取指阶段数据在各组件之间的流动：

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#1a1a2e')

titles = [
    'Step 1: MAR \u2190 [PC]',
    'Step 2: PC \u2190 PC + 1',
    'Step 3: MDR \u2190 [[MAR]]',
    'Step 4: CIR \u2190 [MDR]',
]

highlights = [
    {'PC': ('100', True), 'MAR': ('100', True), 'MDR': ('', False), 'CIR': ('', False), 'MEM': ('LDD 200', False)},
    {'PC': ('101', True), 'MAR': ('100', False), 'MDR': ('', False), 'CIR': ('', False), 'MEM': ('LDD 200', False)},
    {'PC': ('101', False), 'MAR': ('100', False), 'MDR': ('LDD 200', True), 'CIR': ('', False), 'MEM': ('LDD 200', True)},
    {'PC': ('101', False), 'MAR': ('100', False), 'MDR': ('LDD 200', False), 'CIR': ('LDD 200', True), 'MEM': ('LDD 200', False)},
]

arrow_pairs = [
    [('PC', 'MAR')],
    [],
    [('MEM', 'MDR')],
    [('MDR', 'CIR')],
]

positions = {'PC': (1, 3), 'MAR': (3, 3), 'MDR': (3, 1), 'CIR': (1, 1), 'MEM': (5, 2)}

for idx, ax in enumerate(axes.flat):
    ax.set_facecolor('#16213e')
    ax.set_xlim(0, 7)
    ax.set_ylim(0, 5)
    ax.axis('off')
    ax.set_title(titles[idx], fontsize=11, fontweight='bold', color='white', pad=8)
    
    h = highlights[idx]
    for name, (x, y) in positions.items():
        val, is_active = h[name]
        ec = '#e94560' if is_active else '#555555'
        lw_val = 3 if is_active else 1.5
        w_box = 1.8 if name == 'MEM' else 1.4
        box = mpatches.FancyBboxPatch((x-w_box/2, y-0.4), w_box, 0.8,
            boxstyle='round,pad=0.08', facecolor='#0f3460', edgecolor=ec, linewidth=lw_val)
        ax.add_patch(box)
        display_name = 'Memory' if name == 'MEM' else name
        ax.text(x, y+0.4, display_name, fontsize=9, fontweight='bold',
            color='white', ha='center', va='center')
        ax.text(x, y-0.15, val, fontsize=8, color='#ffd700', ha='center', va='center')
    
    for src, dst in arrow_pairs[idx]:
        sx, sy = positions[src]
        dx, dy = positions[dst]
        ax.annotate('', xy=(dx, dy+0.4), xytext=(sx, sy-0.4),
            arrowprops=dict(arrowstyle='->', color='#e94560', lw=2.5))

fig.suptitle('Fetch Phase: Data Flow Between Registers',
    fontsize=14, fontweight='bold', color='white', y=0.98)
plt.tight_layout()
plt.show()

## 15. 练习题

### 题目1
写出完整的取指阶段的四个步骤（使用RTN表示法）。

### 题目2
解释在取指阶段，为什么PC要在步骤2就递增（而不是在执行阶段之后）？

### 题目3
对于指令 `ADD 150`，写出其执行阶段的RTN表示法。

### 题目4
解释以下概念：
- (a) 什么是中断？
- (b) 为什么中断机制比轮询 (polling) 更高效？
- (c) 描述中断发生时的处理步骤。

### 题目5
跟踪以下程序的执行过程，写出每条指令执行后ACC的值：
```
地址100: LDM #5    // ACC = ?
地址101: ADD 200   // ACC = ?
地址102: SUB 201   // ACC = ?
地址103: STO 202   // ACC = ?

地址200: 10
地址201: 3
地址202: 0
```

In [ ]:
# \u7ec3\u4e60\u9898\u7b54\u6848\u68c0\u67e5

print('\u9898\u76ee5 \u7b54\u6848\u8ddf\u8e2a:')
print('LDM #5  -> ACC = 5    (\u7acb\u5373\u5bfb\u5740\uff0c\u76f4\u63a5\u52a0\u8f7d\u50065)')
print('ADD 200 -> ACC = 5 + 10 = 15  (\u52a0\u4e0a\u5730\u5740200\u7684\u503c)')
print('SUB 201 -> ACC = 15 - 3 = 12  (\u51cf\u53bb\u5730\u5740201\u7684\u503c)')
print('STO 202 -> ACC = 12   (ACC\u4e0d\u53d8\uff0c\u503c\u5b58\u5165\u5730\u5740202)')
print(f'\n\u6700\u7ec8: memory[202] = 12')

## 16. 拓展：流水线技术 (Pipelining)

现代CPU并不是等一条指令完全执行完才开始下一条。它们使用**流水线**技术：

```
时间 →   T1    T2    T3    T4    T5    T6
指令1: [Fetch][Decode][Execute]
指令2:       [Fetch][Decode][Execute]
指令3:             [Fetch][Decode][Execute]
指令4:                   [Fetch][Decode][Execute]
```

就像工厂流水线：一辆车在喷漆时，另一辆已经在组装了。
这样CPU在同一时刻可以处理多条指令的不同阶段，大大提高了效率！

> 这就是为什么现代CPU每秒能执行数十亿条指令——不是因为每条指令执行得有多快，而是因为很多指令在**并行处理**。

## 17. 拓展知识：现实中的CPU

- **Intel Core i9**：14核心，最高6.0 GHz，支持超线程
- **Apple M3**：采用ARM架构，混合大小核设计
- **现代CPU的流水线**可能有15-20个阶段（不只是3个）
- **乱序执行 (Out-of-Order Execution)**：CPU可以调整指令执行顺序以提高效率
- **分支预测 (Branch Prediction)**：CPU猜测跳转指令的结果，提前执行

---

**下一节预告**：我们将学习如何用汇编语言直接编写CPU指令——汇编语言 (Assembly Language)！